# 🏗️ Notebook 1: Key-Value Store — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A **distributed key-value store** like DynamoDB / Cassandra: `get(k)` / `put(k, v)` /
`delete(k)` at large scale, across many machines, surviving node failures.

### Functional requirements
- `put(key, value)`, `get(key)`, `delete(key)`
- Tunable consistency (eventual vs strong)
- Data must survive single-node failure

### Non-functional
- **Horizontally scalable**: add nodes, capacity grows.
- **High availability**: `put/get` should work even if some nodes are down.
- **Low latency**: <10ms p99 for small values.
- **Durable**: once ack'd, don't lose the write.

### Key questions
1. **How do we pick which node owns key `k`?** — *Partitioning*
2. **How do we survive a node failure?** — *Replication*
3. **How do clients read when replicas disagree?** — *Consistency*


## Consistent hashing (partitioning)

Naive `hash(k) % N` is a disaster when N changes (e.g., node added/removed): *every* key
potentially moves. Consistent hashing limits the damage.

```
     0 ─────── 2^32 ─── (the ring, positions 0..2^32-1)

  place each node at hash(node_id) on the ring
  place each key  at hash(key)     on the ring
  key is owned by the FIRST node clockwise from its position.

        (N1)
         ●
        /
       /          ● (N4)
      /
 ────●            ●────   ring
      \
       \
        ● (N2)   ● (N3)

  adding N5 steals only a slice of one neighbor's keys.
```

### Virtual nodes
A physical node advertises many ring positions (vnodes) so load is balanced more evenly
and rebalance granularity is fine.


## Replication & consistency

For each key, write to **N** replicas (e.g., N=3). Two tunable knobs:

- **W** = min replicas that must acknowledge a write
- **R** = min replicas that must respond to a read

**Strong consistency guarantee**: `R + W > N` → the read set must intersect the write set.

| N | W | R | Meaning |
|---|---|---|---|
| 3 | 1 | 1 | "available, eventually consistent" (Dynamo default) |
| 3 | 2 | 2 | Quorum; survives 1 replica outage |
| 3 | 3 | 1 | Fast reads, slow/blocking writes |

### CAP tradeoff in plain English
During a network partition between replicas, you get to pick two of:
- **C**onsistency (all readers see the latest)
- **A**vailability (every request gets a response)
- **P**artition tolerance (mandatory — networks *will* partition)
